# GridPulse UK — Environment Setup

This notebook validates the Databricks runtime, Spark session, Unity Catalog context,
and outbound connectivity to the NESO Carbon Intensity API.

## Objectives

1. Confirm that serverless compute is available.
2. Inspect the Spark and Databricks environment.
3. Confirm the current Unity Catalog context.
4. Test access to the NESO Carbon Intensity API.

In [0]:
print(f"Spark version: {spark.version}")
try:
    app_name = spark.conf.get("spark.app.name")
except Exception:
    app_name = "N/A (serverless)"
print(f"Application name: {app_name}")

In [0]:
context_df = spark.sql("""
    SELECT
        current_user() AS current_user,
        current_catalog() AS current_catalog,
        current_schema() AS current_schema
""")

display(context_df)

In [0]:
import requests

API_URL = "https://api.carbonintensity.org.uk/intensity"

response = requests.get(
    API_URL,
    timeout=30
)

response.raise_for_status()

payload = response.json()

print(f"HTTP status: {response.status_code}")
print(f"Top-level keys: {list(payload.keys())}")
print(f"Number of data records: {len(payload['data'])}")

display(payload["data"])

## Unity Catalog Foundation

GridPulse uses a medallion architecture:

- **Bronze:** Raw and minimally processed source data.
- **Silver:** Cleaned, validated, deduplicated and conformed data.
- **Gold:** Business-ready facts, dimensions and aggregates.
- **Operations:** Pipeline audit logs, quality results and processing metadata.

Raw source files are stored in a Unity Catalog managed volume before being
processed into Delta tables.

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS gridpulse
COMMENT 'Governed lakehouse for the GridPulse UK energy and carbon-intensity platform';

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gridpulse.bronze
COMMENT 'Raw and minimally processed source data';

CREATE SCHEMA IF NOT EXISTS gridpulse.silver
COMMENT 'Validated, deduplicated and conformed energy data';

CREATE SCHEMA IF NOT EXISTS gridpulse.gold
COMMENT 'Business-ready dimensional models and analytical aggregates';

CREATE SCHEMA IF NOT EXISTS gridpulse.operations
COMMENT 'Pipeline audit logs, quality results and processing metadata';

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS gridpulse.bronze.raw_landing
COMMENT 'Immutable landing area for raw NESO API responses and source files';

In [0]:
%sql
SHOW SCHEMAS IN gridpulse;

In [0]:
%sql
SHOW VOLUMES IN gridpulse.bronze;

In [0]:
from datetime import datetime, timezone
import json
import requests

API_URL = "https://api.carbonintensity.org.uk/intensity"

ingested_at = datetime.now(timezone.utc)

response = requests.get(
    API_URL,
    timeout=30
)
response.raise_for_status()

raw_payload = response.json()

date_partition = ingested_at.strftime("%Y/%m/%d")
file_timestamp = ingested_at.strftime("%Y%m%dT%H%M%SZ")

landing_directory = (
    f"/Volumes/gridpulse/bronze/raw_landing/"
    f"carbon_intensity/national/{date_partition}"
)

file_path = (
    f"{landing_directory}/"
    f"carbon_intensity_{file_timestamp}.json"
)

raw_envelope = {
    "source_system": "neso_carbon_intensity_api",
    "source_url": API_URL,
    "ingested_at_utc": ingested_at.isoformat(),
    "payload": raw_payload
}

dbutils.fs.mkdirs(landing_directory)

dbutils.fs.put(
    file_path,
    json.dumps(raw_envelope),
    overwrite=False
)

print(f"Raw file written successfully: {file_path}")

## Bronze Carbon-Intensity Table

The raw API files are preserved unchanged in the landing volume.

This section:

1. Reads all national carbon-intensity files recursively.
2. Applies an explicit schema.
3. Converts the nested API response into one row per electricity period.
4. Adds technical lineage columns.
5. generates a deterministic record identifier.
6. Loads records idempotently into a Bronze Delta table.

In [0]:
RAW_NATIONAL_PATH = (
    "/Volumes/gridpulse/bronze/raw_landing/"
    "carbon_intensity/national/"
)

BRONZE_TABLE = "gridpulse.bronze.carbon_intensity"

In [0]:
from pyspark.sql.types import (
    ArrayType,
    IntegerType,
    StringType,
    StructField,
    StructType
)

intensity_schema = StructType([
    StructField("forecast", IntegerType(), True),
    StructField("actual", IntegerType(), True),
    StructField("index", StringType(), True)
])

period_schema = StructType([
    StructField("from", StringType(), True),
    StructField("to", StringType(), True),
    StructField("intensity", intensity_schema, True)
])

payload_schema = StructType([
    StructField(
        "data",
        ArrayType(period_schema),
        True
    )
])

raw_envelope_schema = StructType([
    StructField("source_system", StringType(), False),
    StructField("source_url", StringType(), False),
    StructField("ingested_at_utc", StringType(), False),
    StructField("payload", payload_schema, True)
])

In [0]:
raw_df = (
    spark.read
    .format("json")
    .schema(raw_envelope_schema)
    .option("recursiveFileLookup", "true")
    .load(RAW_NATIONAL_PATH)
)

raw_df.printSchema()
display(raw_df)

In [0]:
display(
    raw_df.select(
        "source_system",
        "ingested_at_utc",
        "payload.data"
    )
)

In [0]:
from pyspark.sql.functions import col, explode_outer

exploded_df = raw_df.withColumn(
    "period",
    explode_outer(col("payload.data"))
)

display(
    exploded_df.select(
        "source_system",
        "ingested_at_utc",
        "period"
    )
)

In [0]:
simple_bronze_df = exploded_df.select(
    col("source_system"),
    col("ingested_at_utc"),
    col("period.from").alias("period_start"),
    col("period.to").alias("period_end"),
    col("period.intensity.forecast").alias("forecast"),
    col("period.intensity.actual").alias("actual"),
    col("period.intensity.index").alias("intensity_index")
)

display(simple_bronze_df)

In [0]:
from pyspark.sql.functions import col, to_timestamp

bronze_df = simple_bronze_df.select(
    col("source_system"),

    to_timestamp(
        col("ingested_at_utc")
    ).alias("ingested_at_utc"),

    to_timestamp(
        col("period_start"),
        "yyyy-MM-dd'T'HH:mmX"
    ).alias("period_start_utc"),

    to_timestamp(
        col("period_end"),
        "yyyy-MM-dd'T'HH:mmX"
    ).alias("period_end_utc"),

    col("forecast").alias("forecast_gco2_per_kwh"),

    col("actual").alias("actual_gco2_per_kwh"),

    col("intensity_index")
)

display(bronze_df)
bronze_df.printSchema()

In [0]:
from pyspark.sql.functions import (
    col,
    concat_ws,
    current_timestamp,
    sha2
)

bronze_ready_df = (
    bronze_df
    .withColumn(
        "record_id",
        sha2(
            concat_ws(
                "||",
                col("source_system"),
                col("ingested_at_utc").cast("string"),
                col("period_start_utc").cast("string"),
                col("period_end_utc").cast("string")
            ),
            256
        )
    )
    .withColumn(
        "loaded_at_utc",
        current_timestamp()
    )
    .select(
        "record_id",
        "source_system",
        "ingested_at_utc",
        "period_start_utc",
        "period_end_utc",
        "forecast_gco2_per_kwh",
        "actual_gco2_per_kwh",
        "intensity_index",
        "loaded_at_utc"
    )
)

display(bronze_ready_df)

In [0]:
total_records = bronze_ready_df.count()

missing_record_ids = (
    bronze_ready_df
    .filter(col("record_id").isNull())
    .count()
)

missing_period_starts = (
    bronze_ready_df
    .filter(col("period_start_utc").isNull())
    .count()
)

missing_period_ends = (
    bronze_ready_df
    .filter(col("period_end_utc").isNull())
    .count()
)

print(f"Total records: {total_records}")
print(f"Missing record IDs: {missing_record_ids}")
print(f"Missing period starts: {missing_period_starts}")
print(f"Missing period ends: {missing_period_ends}")

In [0]:
assert total_records > 0, (
    "Validation failed: no records were extracted."
)

assert missing_record_ids == 0, (
    "Validation failed: record_id contains null values."
)

assert missing_period_starts == 0, (
    "Validation failed: period_start_utc contains null values."
)

assert missing_period_ends == 0, (
    "Validation failed: period_end_utc contains null values."
)

print("Bronze validation passed.")

In [0]:
BRONZE_TABLE = "gridpulse.bronze.carbon_intensity"

In [0]:
from delta.tables import DeltaTable

if not spark.catalog.tableExists(BRONZE_TABLE):

    (
        bronze_ready_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(BRONZE_TABLE)
    )

    print(f"Created Delta table: {BRONZE_TABLE}")

else:

    bronze_delta_table = DeltaTable.forName(
        spark,
        BRONZE_TABLE
    )

    (
        bronze_delta_table.alias("target")
        .merge(
            bronze_ready_df.alias("source"),
            "target.record_id = source.record_id"
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"Merged data into: {BRONZE_TABLE}")

In [0]:
%sql
SELECT *
FROM gridpulse.bronze.carbon_intensity
ORDER BY ingested_at_utc DESC;

In [0]:
%sql
SELECT COUNT(*) AS bronze_record_count
FROM gridpulse.bronze.carbon_intensity;

In [0]:
%sql
DESCRIBE HISTORY gridpulse.bronze.carbon_intensity;

In [0]:
%sql
SELECT COUNT(*) AS bronze_record_count
FROM gridpulse.bronze.carbon_intensity;

In [0]:
%sql
COMMENT ON TABLE gridpulse.bronze.carbon_intensity IS
'Immutable national carbon-intensity snapshots ingested from the NESO Carbon Intensity API';

ALTER TABLE gridpulse.bronze.carbon_intensity
SET TBLPROPERTIES (
    'quality' = 'bronze',
    'source.system' = 'NESO Carbon Intensity API',
    'data.owner' = 'GridPulse'
);

## Day 1 Status

- Unity Catalog structure created
- Raw NESO API responses landed in a managed volume
- Nested JSON flattened with PySpark
- Bronze Delta table created
- Idempotent merge validated